# PA / PB1 peptide design — protocol comparison

Loads ColabDesign generation/optimisation results and Boltz structural validation,
then compares protocols on both sets of metrics.

In [ ]:
import json, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', palette='tab10')
pd.set_option('display.float_format', '{:.3f}'.format)

## 1. Paths

In [ ]:
import sys, os, pathlib

# Load .env from project root (one level up from notebooks/)
_env_path = pathlib.Path('..') / '.env'
if _env_path.exists():
    for _line in _env_path.read_text().splitlines():
        _line = _line.strip()
        if _line.startswith('export '):
            _line = _line[7:]
        if '=' in _line and not _line.startswith('#'):
            _k, _v = _line.split('=', 1)
            os.environ.setdefault(_k.strip(), _v.strip())

sys.path.append('..')
import config

PRJ_DIR    = pathlib.Path('..').resolve()
SCRIPT_DIR = PRJ_DIR / 'notebooks'
COLAB_DIR  = pathlib.Path(config.RESULTS_DIR)
BOLTZ_DIR  = pathlib.Path(config.BOLTZ_RESULTS_DIR)

print(f'COLAB_DIR  exists: {COLAB_DIR.exists()}  ({len(list(COLAB_DIR.glob("*.json")))} JSON files)')
print(f'BOLTZ_DIR  exists: {BOLTZ_DIR.exists()}')

## 2. Load ColabDesign results

In [ ]:
def load_colab_results(colab_dir: pathlib.Path) -> pd.DataFrame:
    rows = []
    for jf in sorted(colab_dir.rglob('*.json')):
        proto = jf.stem
        data  = json.loads(jf.read_text())
        for i, entry in enumerate(data.get('results', [])):
            rows.append({
                'name'       : f'{proto}_{i}',
                'protocol'   : proto,
                'idx'        : i,
                'seq'        : entry.get('seq', ''),
                'loss_total' : entry.get('loss_total'),
                'loss_af'    : entry.get('loss_af'),
                'energy'     : entry.get('energy'),
                'colab_iptm' : entry.get('i_ptm'),
                'colab_ptm'  : entry.get('ptm'),
                'colab_plddt': entry.get('plddt'),
            })
    if not rows:
        raise RuntimeError(f'No results found in {colab_dir} — check COLAB_DIR path above.')
    df = pd.DataFrame(rows)
    print(f'Loaded {len(df)} ColabDesign entries from {df["protocol"].nunique()} protocols')
    return df

df_colab = load_colab_results(COLAB_DIR)
df_colab.head()

## 3. Load Boltz results

In [ ]:
BOLTZ_COLS = [
    'name', 'boltz_confidence', 'boltz_ptm', 'boltz_iptm',
    'boltz_plddt', 'boltz_iplddt', 'boltz_pde', 'boltz_ipde',
    'boltz_ptm_PA', 'boltz_ptm_pep', 'boltz_iptm_cross',
]

def load_boltz_results(boltz_dir: pathlib.Path) -> pd.DataFrame:
    """
    Expected output layout created by run_boltz_peptides.sh:
      boltz_dir/
        boltz_results_pa_<name>/
          predictions/
            pa_<name>/
              confidence_pa_<name>_model_0.json
    Returns an empty DataFrame with the correct schema if no results are found.
    """
    rows = []
    for result_dir in sorted(boltz_dir.glob('boltz_results_pa_*')):
        name = result_dir.name.removeprefix('boltz_results_pa_')
        conf_glob = list(result_dir.glob(f'predictions/pa_{name}/confidence_pa_{name}_model_0.json'))
        if not conf_glob:
            print(f'  [missing] {result_dir.name}')
            continue
        conf = json.loads(conf_glob[0].read_text())
        rows.append({
            'name'             : name,
            'boltz_confidence' : conf.get('confidence_score'),
            'boltz_ptm'        : conf.get('ptm'),
            'boltz_iptm'       : conf.get('iptm'),
            'boltz_plddt'      : conf.get('complex_plddt'),
            'boltz_iplddt'     : conf.get('complex_iplddt'),
            'boltz_pde'        : conf.get('complex_pde'),
            'boltz_ipde'       : conf.get('complex_ipde'),
            'boltz_ptm_PA'     : conf.get('chains_ptm', {}).get('0'),
            'boltz_ptm_pep'    : conf.get('chains_ptm', {}).get('1'),
            'boltz_iptm_cross' : conf.get('pair_chains_iptm', {}).get('1', {}).get('0'),
        })
    if not rows:
        print('No Boltz results found — returning empty table.')
        return pd.DataFrame(columns=BOLTZ_COLS)
    df = pd.DataFrame(rows)
    print(f'Loaded {len(df)} Boltz entries')
    return df

df_boltz = load_boltz_results(BOLTZ_DIR)
df_boltz.head()

## 4. Merge

In [ ]:
# Average duplicate names in each table, then outer-merge so all entries are kept
def avg_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    num_cols = df.select_dtypes('number').columns.tolist()
    str_cols = [c for c in df.columns if c not in num_cols and c != 'name']
    agg = {c: 'mean' for c in num_cols}
    agg.update({c: 'first' for c in str_cols})
    return df.groupby('name', as_index=False).agg(agg)

df_colab_u = avg_duplicates(df_colab)
df_boltz_u = avg_duplicates(df_boltz)

df = df_colab_u.merge(df_boltz_u, on='name', how='outer')

n_both       = (df['loss_af'].notna() & df['boltz_confidence'].notna()).sum()
n_colab_only = (df['loss_af'].notna() & df['boltz_confidence'].isna()).sum()
n_boltz_only = (df['loss_af'].isna()  & df['boltz_confidence'].notna()).sum()
print(f'{len(df)} total entries | {n_both} both | {n_colab_only} colab-only | {n_boltz_only} boltz-only')
df.head()

## 5. Per-protocol summary statistics

In [ ]:
METRICS = [
    'loss_af', 'energy', 'colab_iptm', 'colab_ptm', 'colab_plddt',
    'boltz_confidence', 'boltz_iptm', 'boltz_plddt', 'boltz_iplddt', 'boltz_iptm_cross',
]

available = [m for m in METRICS if m in df.columns and df[m].notna().any()]

summary = (
    df.groupby('protocol')[available]
    .agg(['mean', 'std', 'min', 'max', 'count'])
    .round(3)
)
summary

## 6. Visualisations
### 6.1 Box plots — ColabDesign metrics

In [ ]:
colab_metrics = ['loss_af', 'energy', 'colab_iptm', 'colab_ptm', 'colab_plddt']
colab_metrics = [m for m in colab_metrics if m in df.columns]

fig, axes = plt.subplots(1, len(colab_metrics), figsize=(4*len(colab_metrics), 5), sharey=False)
for ax, metric in zip(axes, colab_metrics):
    sns.boxplot(data=df, x='protocol', y=metric, ax=ax, order=sorted(df['protocol'].unique()))
    ax.set_title(metric)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
    plt.setp(ax.get_xticklabels(), ha='right', rotation_mode='anchor')
fig.suptitle('ColabDesign metrics by protocol', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 6.2 Box plots — Boltz metrics

In [ ]:
boltz_metrics = ['boltz_confidence', 'boltz_iptm', 'boltz_iptm_cross', 'boltz_plddt', 'boltz_iplddt']
boltz_metrics = [m for m in boltz_metrics if m in df.columns and df[m].notna().any()]

if boltz_metrics:
    fig, axes = plt.subplots(1, len(boltz_metrics), figsize=(4*len(boltz_metrics), 5), sharey=False)
    if len(boltz_metrics) == 1:
        axes = [axes]
    for ax, metric in zip(axes, boltz_metrics):
        sns.boxplot(data=df, x='protocol', y=metric, ax=ax, order=sorted(df['protocol'].unique()))
        ax.set_title(metric)
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=45)
        plt.setp(ax.get_xticklabels(), ha='right', rotation_mode='anchor')
    fig.suptitle('Boltz validation metrics by protocol', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No Boltz results available yet.')

### 6.3 ColabDesign vs Boltz correlation

In [ ]:
pairs = [
    ('colab_iptm',  'boltz_iptm'),
    ('colab_iptm',  'boltz_iptm_cross'),
    ('colab_plddt', 'boltz_plddt'),
    ('energy',      'boltz_confidence'),
]
pairs = [(x, y) for x, y in pairs
         if x in df.columns and y in df.columns and df[y].notna().any()]

if pairs:
    fig, axes = plt.subplots(1, len(pairs), figsize=(5*len(pairs), 4))
    if len(pairs) == 1:
        axes = [axes]
    protos  = sorted(df.protocol.unique())
    palette = dict(zip(protos, sns.color_palette('tab10', len(protos))))
    for ax, (xcol, ycol) in zip(axes, pairs):
        for proto, grp in df.dropna(subset=[xcol, ycol]).groupby('protocol'):
            ax.scatter(grp[xcol], grp[ycol], label=proto, alpha=0.7,
                       color=palette[proto], s=30)
        ax.set_xlabel(xcol); ax.set_ylabel(ycol)
        r = df[[xcol, ycol]].dropna().corr().iloc[0, 1]
        ax.set_title(f'r = {r:.2f}')
    handles = [plt.Line2D([0],[0], marker='o', color='w',
                           markerfacecolor=palette[p], label=p, markersize=8)
               for p in protos]
    axes[-1].legend(handles=handles, bbox_to_anchor=(1.05,1), loc='upper left', fontsize=8)
    fig.suptitle('ColabDesign vs Boltz correlations', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('No Boltz results to correlate yet.')

### 6.4 Protocol ranking — mean boltz_confidence

In [ ]:
if df['boltz_confidence'].notna().any():
    rank = (
        df.groupby('protocol')['boltz_iptm_cross']
        .agg(['mean', 'std', 'count', 'max'])
        .sort_values('mean', ascending=False)
    )
    fig, ax = plt.subplots(figsize=(7, 0.6*len(rank)+1))
    colors = sns.color_palette('RdYlGn', len(rank))
    ax.barh(rank.index, rank['mean'], xerr=rank['std'], align='center',
            color=colors[::-1], capsize=4)
    ax.scatter(rank['max'], rank.index, color='black', zorder=5,
               s=60, marker='D', label='max')
    ax.legend(loc='lower right', fontsize=9)
    ax.set_xlabel('boltz_iptm_cross')
    ax.set_title('Protocol ranking (Boltz iptm cross)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    display(rank)
else:
    print('No Boltz results available yet.')

### 6.5 Heatmap — mean metrics per protocol

In [ ]:
heat_cols = [m for m in [
    'loss_af', 'energy', 'colab_iptm', 'colab_plddt',
    'boltz_confidence', 'boltz_iptm', 'boltz_iptm_cross', 'boltz_plddt',
] if m in df.columns and df[m].notna().any()]

heat   = df.groupby('protocol')[heat_cols].mean()

# For lower-is-better metrics, flip the z-score so high always means good
LOWER_IS_BETTER = {'loss_af', 'energy', 'loss_total'}
heat_z = (heat - heat.mean()) / heat.std()
for col in heat_z.columns:
    if col in LOWER_IS_BETTER:
        heat_z[col] = -heat_z[col]

fig, ax = plt.subplots(figsize=(len(heat_cols)*1.4+1, len(heat)*0.6+1))
sns.heatmap(heat_z, annot=heat.round(3), fmt='.3f', cmap='Reds',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'goodness z-score (dark red = good)'})
ax.set_title('Per-protocol mean metrics (dark red = good, white = bad)')
plt.tight_layout()
plt.show()

## 7. Top candidates

In [ ]:
sort_col = 'boltz_iptm_cross' if df.get('boltz_iptm_cross', pd.Series(dtype=float)).notna().any() else 'colab_iptm'
print(f'Ranking by: {sort_col}')

top = (
    df.dropna(subset=[sort_col])
    .sort_values(sort_col, ascending=False)
    .head(20)[['name', 'protocol', 'seq', 'colab_iptm', 'colab_plddt',
               'boltz_confidence', 'boltz_iptm', 'boltz_iptm_cross', 'boltz_plddt']]
    .reset_index(drop=True)
)
display(top)

In [ ]:
OUT = SCRIPT_DIR / 'results' / 'top_candidates.csv'
top.to_csv(OUT, index=False)
print(f'Saved to {OUT}')

## 8. Sequence space — PCA + t-SNE

Each peptide is encoded as an integer vector of length 15 (one integer per position, 1–20 for the 20 standard amino acids).  
PCA is applied first to decorrelate and optionally reduce dimensionality; t-SNE then maps to 2-D.

### 8.1 Parameters

In [ ]:
# ── Encoding ──────────────────────────────────────────────────────────────────
AA_ORDER   = 'ACDEFGHIKLMNPQRSTVWY'   # standard 20 AA → integers 1–20
AA_TO_INT  = {aa: i+1 for i, aa in enumerate(AA_ORDER)}

# ── PCA ───────────────────────────────────────────────────────────────────────
N_PCA       = 10 #10     # PCA components fed into t-SNE (None → skip PCA, use raw vectors)

# ── t-SNE ─────────────────────────────────────────────────────────────────────
PERPLEXITY  = 30     # typical range 5–50; try 10, 20, 30, 50
N_ITER      = 1000   # optimisation steps (≥250)
LEARNING_RATE = 'auto'  # 'auto' or a float (200 is a common default)
RANDOM_STATE  = 42

### 8.2 Encode sequences

In [ ]:
def encode_sequence(seq: str, aa_to_int: dict, length: int = 15) -> np.ndarray:
    """Integer-encode a peptide sequence.  Unknown AAs are mapped to 0."""
    vec = np.zeros(length, dtype=np.float32)
    for i, aa in enumerate(seq[:length]):
        vec[i] = aa_to_int.get(aa, 0)
    return vec

SEQ_LEN = df['seq'].str.len().mode()[0]   # expected peptide length
print(f'Sequence length detected: {SEQ_LEN}')

# Filter to well-formed sequences only
mask  = df['seq'].str.len() == SEQ_LEN
df_ok = df[mask].reset_index(drop=True)
print(f'Sequences kept: {len(df_ok)}/{len(df)}')

X_raw = np.stack([encode_sequence(s, AA_TO_INT, SEQ_LEN) for s in df_ok['seq']])
print(f'Feature matrix: {X_raw.shape}  (values 0–20, 0 = unknown)')

### 8.3 PCA — scree plot and projection

In [ ]:
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)

evr     = pca_full.explained_variance_ratio_
cumevr  = np.cumsum(evr)
n_comps = len(evr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar(range(1, n_comps+1), evr * 100, color='steelblue')
ax1.set_xlabel('PC'); ax1.set_ylabel('Explained variance (%)')
ax1.set_title('Scree plot')

ax2.plot(range(1, n_comps+1), cumevr * 100, marker='o', color='steelblue', ms=4)
if N_PCA is not None:
    ax2.axvline(N_PCA, color='tomato', linestyle='--',
                label=f'N_PCA={N_PCA}  ({cumevr[N_PCA-1]*100:.1f}% var)')
    ax2.legend()
ax2.set_xlabel('Number of PCs'); ax2.set_ylabel('Cumulative explained variance (%)')
ax2.set_title('Cumulative variance')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.suptitle('PCA on integer-encoded sequences', fontsize=13)
plt.tight_layout()
plt.show()

# Reduce to N_PCA components for t-SNE input
if N_PCA is not None and N_PCA < X_scaled.shape[1]:
    pca_reduce = PCA(n_components=N_PCA, random_state=RANDOM_STATE)
    X_pca = pca_reduce.fit_transform(X_scaled)
    print(f'PCA: {X_scaled.shape[1]} → {N_PCA} dims  '
          f'({cumevr[N_PCA-1]*100:.1f}% variance retained)')
else:
    X_pca = X_scaled
    print('PCA skipped — using full scaled matrix')

### 8.4 t-SNE

In [ ]:
tsne = TSNE(
    n_components  = 2,
    perplexity    = min(PERPLEXITY, len(X_pca) - 1),
    #n_iter        = N_ITER,
    learning_rate = LEARNING_RATE,
    random_state  = RANDOM_STATE,
    init          = 'pca',   # warm-start from PCA for stability
    verbose       = 1,
)
X_tsne = tsne.fit_transform(X_pca)
df_ok  = df_ok.assign(tsne_x=X_tsne[:, 0], tsne_y=X_tsne[:, 1])
print('t-SNE done. KL divergence:', round(tsne.kl_divergence_, 4))

### 8.5 t-SNE map — coloured by protocol

In [ ]:
protos  = sorted(df_ok['protocol'].unique())
palette = dict(zip(protos, sns.color_palette('tab10', len(protos))))
markers = ['o', 's', 'D', 'H', ',', 'P', 'X', '*', '>', '<', '^', 'v',]   # one per protocol
mmap    = dict(zip(protos, markers))

fig, ax = plt.subplots(figsize=(8, 7))
for proto, grp in df_ok.groupby('protocol'):
    ax.scatter(
        grp['tsne_x'], grp['tsne_y'],
        color  = palette[proto],
        marker = mmap.get(proto, 'o'),
        s      = 60,
        alpha  = 0.75,
        label  = proto,
        edgecolors='none',
    )

ax.legend(title='protocol', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.set_title(
    f't-SNE of designed sequences (perplexity={PERPLEXITY}, '
    f'n_iter={N_ITER}, PCA→{N_PCA}d)'
)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

### 8.6 t-SNE map — coloured by quality metric

Same layout, but point colour encodes a continuous quality score (uses `boltz_iptm_cross` if available, else `colab_iptm`).  
This lets you see whether high-quality sequences cluster in a specific region of sequence space.

In [ ]:
quality_col = (
    'boltz_iptm_cross'
    if 'boltz_iptm_cross' in df_ok.columns and df_ok['boltz_iptm_cross'].notna().any()
    else 'colab_iptm'
)
print(f'Quality metric: {quality_col}')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
colormaps=['RdYlGn', 'RdYlGn_r']

for ax, col in zip(axes, [quality_col, 'energy']):
    vals = df_ok[col]
    sc = ax.scatter(
        df_ok['tsne_x'], df_ok['tsne_y'],
        c      = vals,
        cmap   = colormaps[0] if col == quality_col else colormaps[1],
        s      = 60,
        alpha  = 0.85,
        vmin   = vals.quantile(0.05),
        vmax   = vals.quantile(0.95),
        edgecolors='none',
    )
    plt.colorbar(sc, ax=ax, label=col, fraction=0.04)
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    ax.set_title(col)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('Sequence space coloured by quality metrics', fontsize=13)
plt.tight_layout()
plt.show()